# PetTriage — Feature Engineering

**Input:** `data/processed/training_data.csv` (2,200 rows, validated in notebook 03)

**Output:** `data/processed/train.csv`, `val.csv`, `test.csv` — three separate piles of the
same data, reshaped from 23 human-readable columns into model-ready number columns.

**Why this notebook exists:** most ML models can only do math on numbers — they can't read
`"Vomiting"` or `"3 weeks"` directly. This notebook turns the validated-but-still-messy
`training_data.csv` into something a model can be trained on, while being careful not to let
the model "cheat" by seeing data it's supposed to be tested on later (more on this below).

**Steps:**
1. Split the data into train / validation / test piles (stratified — see Step 1)
2. Consolidate `Symptom_1-4` + the 9 legacy flag columns into one multi-hot symptom representation
3. Parse `Duration` free text into a numeric `Duration_Days` feature
4. Check whether `Breed` is actually worth keeping, before encoding it
5. One-hot encode `Animal_Type`, `Gender`
6. Scale numeric features
7. Label-encode both targets (`Disease_Category`, `Urgency`)
8. Save `train.csv` / `val.csv` / `test.csv`


In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

df = pd.read_csv('../data/processed/training_data.csv')
print('Loaded:', df.shape)


Loaded: (2200, 23)


## Step 1: Split into Train / Validation / Test

**In plain terms:** think of the 2,200 cases as a deck of cards. Before the model touches
anything, we deal it into three separate piles:
- **Train (70%, ~1,540 rows)** — what the model actually studies.
- **Validation (15%, ~330 rows)** — a practice quiz, used while we're still tuning things.
- **Test (15%, ~330 rows)** — locked away, opened only once at the very end, for an honest
  final grade.

**Why not just shuffle and cut randomly?** Some Category+Urgency combinations are rare —
e.g. `Skin/Fungal + Emergency` is only 10 cases out of all 2,200. A blind random cut could
dump most of those 10 into train and leave almost none for testing, so we'd never really know
if the model handles that case at all.

**"Stratified" split** = instead of a blind random cut, force every pile to get a fair,
proportional share of *every* Category+Urgency combination, including the rare ones. If a
combination is 0.45% of the whole dataset, it stays ~0.45% of each pile too.

**Why a combined key, not just Urgency alone?** Category and Urgency aren't independent —
notebook 03 showed Trauma/Poisoning is mostly Emergency, Skin/Fungal mostly isn't. If we only
stratified on Urgency, we could still accidentally cluster all the Trauma cases into one pile.
So we glue the two labels together first (e.g. `"Trauma / Poisoning_Emergency"`) and stratify
on *that*, guaranteeing every specific combination is represented fairly everywhere.

This split happens **before** every other step in this notebook — see Step 6 for why that
ordering matters.


In [2]:
df['strat_key'] = df['Disease_Category'] + '_' + df['Urgency']

print('Smallest strata (watch for very small groups):')
print(df['strat_key'].value_counts().sort_values().head(5))

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['strat_key'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['strat_key'], random_state=42)

for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name}: {part.shape}')


Smallest strata (watch for very small groups):
strat_key
Skin / Fungal_Emergency            10
Trauma / Poisoning_Okay            10
Viral Systemic_Okay                20
Eye / Ear_Emergency                30
Endocrine / Metabolic_Emergency    30
Name: count, dtype: int64
train: (1540, 24)
val: (330, 24)
test: (330, 24)


In [3]:
# Sanity check: did stratification actually preserve Urgency proportions across splits?
print('Urgency proportions per split (should be near-identical):')
for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(name, part['Urgency'].value_counts(normalize=True).round(3).to_dict())

splits = {
    'train': train_df.drop(columns=['strat_key']).copy(),
    'val': val_df.drop(columns=['strat_key']).copy(),
    'test': test_df.drop(columns=['strat_key']).copy(),
}


Urgency proportions per split (should be near-identical):
train {'Monitor': 0.473, 'Okay': 0.268, 'Emergency': 0.259}
val {'Monitor': 0.476, 'Okay': 0.264, 'Emergency': 0.261}
test {'Monitor': 0.47, 'Okay': 0.273, 'Emergency': 0.258}


## Step 2: Symptom Consolidation

`Symptom_1-4` are four text slots in an arbitrary order — there's no real "1st symptom" vs
"4th symptom," the generator just listed however many it produced. Worse: only 9 of the 31
distinct symptom strings that show up in those slots (things like `Fever`, `Lethargy`,
`Seizures`, `Weight Loss`) have a dedicated Yes/No flag column — the rest were sitting there
as unused text.

**Fix — "multi-hot" encoding:** build one `Symptom_<name>` binary (1/0) column per distinct
symptom (31 total), set to 1 if that symptom appears *anywhere* in the row's 4 slots. This
doesn't care about position — "Vomiting" in slot 1 or slot 3 means the same thing — and it
captures all 31 symptom types instead of just the 9 that happened to get a flag column
originally.

We then cross-check the new columns against the 9 old flag columns (expect 0 mismatches,
since notebook 03 already confirmed they agree) and drop the old flags + raw `Symptom_1-4` —
keeping both would just be the same information stored twice, which adds noise without adding
signal.


In [4]:
SYMPTOM_COLS = ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4']
all_symptoms = sorted(s for s in pd.unique(df[SYMPTOM_COLS].values.ravel()) if pd.notna(s))
print(f'{len(all_symptoms)} distinct symptom strings found')

def symptom_col(s):
    return 'Symptom_' + s.replace(' ', '_')

def build_multihot(part):
    part = part.copy()
    for s in all_symptoms:
        part[symptom_col(s)] = part[SYMPTOM_COLS].eq(s).any(axis=1).astype(int)
    return part

for name in splits:
    splits[name] = build_multihot(splits[name])


31 distinct symptom strings found


In [5]:
# Cross-check derived multi-hot vs the 9 legacy flag columns (on full df, before split)
FLAG_TO_SYMPTOMS = {
    'Vomiting': ['Vomiting'],
    'Diarrhea': ['Diarrhea'],
    'Coughing': ['Coughing'],
    'Labored_Breathing': ['Labored Breathing'],
    'Lameness': ['Limping'],
    'Skin_Lesions': ['Skin Lesions', 'Excessive Scratching'],
    'Nasal_Discharge': ['Nasal Discharge'],
    'Eye_Discharge': ['Eye Discharge'],
    'Appetite_Loss': ['Loss of Appetite'],
}

full_multihot = build_multihot(df)
print('Mismatches between derived multi-hot and legacy flags (expect 0 everywhere):')
for flag_col, symptom_names in FLAG_TO_SYMPTOMS.items():
    derived = full_multihot[SYMPTOM_COLS].isin(symptom_names).any(axis=1).astype(int)
    actual = (full_multihot[flag_col] == 'Yes').astype(int)
    print(f'  {flag_col}: {(derived != actual).sum()} mismatches')


Mismatches between derived multi-hot and legacy flags (expect 0 everywhere):
  Vomiting: 0 mismatches
  Diarrhea: 0 mismatches
  Coughing: 0 mismatches
  Labored_Breathing: 0 mismatches
  Lameness: 0 mismatches
  Skin_Lesions: 0 mismatches
  Nasal_Discharge: 0 mismatches
  Eye_Discharge: 0 mismatches
  Appetite_Loss: 0 mismatches


In [6]:
symptom_multihot_cols = [symptom_col(s) for s in all_symptoms]
old_flag_cols = list(FLAG_TO_SYMPTOMS.keys())

for name in splits:
    part = splits[name]
    part['Symptom_Count'] = part[symptom_multihot_cols].sum(axis=1)
    splits[name] = part.drop(columns=SYMPTOM_COLS + old_flag_cols)

print(f'Columns per split after consolidation: {len(splits["train"].columns)}')


Columns per split after consolidation: 42


## Step 3: Duration Parsing

`Duration` is free text (`"2 days"`, `"3 weeks"`, `"A few hours"`, `"Today"`, ...) — 18
distinct strings in inconsistent units (hours/days/weeks/months/years).

**In plain terms:** we could treat these 18 strings as 18 separate categories (like Breed),
but that throws away the fact that `"3 weeks"` is *more* than `"2 days"` — it's not just a
different label, it's a bigger number. So instead we parse each string into a single number:
days elapsed (`"3 weeks"` → `21`, `"A few hours"` → `0.25`). Now a model can learn something
like "risk rises the longer symptoms have persisted" as a smooth relationship, instead of
treating each duration as an unrelated category.


In [7]:
def parse_duration_to_days(s):
    s = str(s).strip().lower()
    if s == 'today':
        return 0.5
    if 'few hours' in s:
        return 0.25
    m = re.match(r'(\d+)\s*(day|week|month|year)', s)
    if not m:
        return None
    n, unit = int(m.group(1)), m.group(2)
    mult = {'day': 1, 'week': 7, 'month': 30, 'year': 365}[unit]
    return float(n * mult)

for name in splits:
    splits[name]['Duration_Days'] = splits[name]['Duration'].apply(parse_duration_to_days)
    unparsed = splits[name]['Duration_Days'].isna().sum()
    if unparsed:
        print(f'WARNING: {unparsed} unparsed Duration values in {name}')
    splits[name] = splits[name].drop(columns=['Duration'])

print('Duration_Days summary (train):')
print(splits['train'][['Duration_Days']].describe())


Duration_Days summary (train):
       Duration_Days
count    1540.000000
mean       32.152273
std        64.929298
min         0.250000
25%         3.000000
50%         7.000000
75%        30.000000
max       365.000000


## Step 4: Is `Breed` Actually Worth Keeping?

Before blindly one-hot encoding `Breed` (25 distinct values — 25 new columns), it's worth
asking: does it actually help predict anything, or is it just adding complexity for no
benefit?

**In plain terms:** we measure this with `Cramér's V` — a score from 0 (no relationship at
all) to 1 (one column perfectly determines the other). As a sanity check, we compute the same
score for `Animal_Type` (Dog vs Cat), which we already believe is clinically meaningful, and
compare.

One trap worth knowing: a statistical test can call a relationship "significant" (a low
p-value) purely because we have 2,200 rows spread across 25 breed categories — with that many
categories, small random patterns from how the synthetic data happened to be generated can
look "detectable" even when they're not practically meaningful. Cramér's V tells us how
*strong* the pattern actually is, not just whether it's technically detectable — that's the
number that should drive the decision.


In [8]:
import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2, p, dof, exp = chi2_contingency(ct)
    n = ct.sum().sum()
    k = min(ct.shape) - 1
    return np.sqrt(chi2 / (n * k)), p

for col in ['Breed', 'Animal_Type']:
    v_cat, p_cat = cramers_v(df[col], df['Disease_Category'])
    v_urg, p_urg = cramers_v(df[col], df['Urgency'])
    print(f'{col} vs Disease_Category: Cramers V={v_cat:.3f} (p={p_cat:.4f})')
    print(f'{col} vs Urgency:          Cramers V={v_urg:.3f} (p={p_urg:.4f})')
    print()


Breed vs Disease_Category: Cramers V=0.116 (p=0.0102)
Breed vs Urgency:          Cramers V=0.127 (p=0.0163)

Animal_Type vs Disease_Category: Cramers V=0.165 (p=0.0000)
Animal_Type vs Urgency:          Cramers V=0.023 (p=0.5526)



**Result:** `Breed` scores ~0.12 against both targets — barely above "no relationship"
(rule of thumb: below ~0.1 is negligible, 0.1-0.3 is weak). `Animal_Type`, despite having only
2 possible values instead of 25, scores *higher* against Disease_Category (~0.17) — a cleaner
signal with far less complexity.

There's also a generalization problem beyond the weak signal: our 25 breeds are only the ones
this particular synthetic batch happened to generate. Real users will type breeds from a list
of hundreds, plus mixed breeds. Even with `handle_unknown='ignore'` protecting us from a
crash, any breed the encoder never saw during training becomes an all-zero row — meaning
`Breed` could only ever help for the exact 25 breeds seen here, never generalizing further.

**Decision: drop `Breed`.** Weak signal + a feature that structurally can't generalize past
the training sample isn't worth 25 columns of complexity. `Gender` is kept — unlike Breed,
there's a direct clinical reason for it (e.g. ovarian conditions only apply to female
animals), independent of what this dataset's association tests say. See ADR-014 in
`docs/DECISIONS.md`.


In [9]:
for name in splits:
    splits[name] = splits[name].drop(columns=['Breed'])

print(f'Columns per split after dropping Breed: {len(splits["train"].columns)}')


Columns per split after dropping Breed: 41


## Step 5: One-Hot Encode `Animal_Type`, `Gender`

Both are still plain text. **One-hot encoding** turns each category into its own 1/0
column — e.g. `Animal_Type` becomes two columns, `Animal_Type_Dog` and `Animal_Type_Cat`,
where exactly one of the two is `1` per row. We do this instead of just numbering categories
(Dog=0, Cat=1) because plain numbers would imply a false order or distance between categories
that doesn't exist.

**Fit on train only:** we learn the list of known categories from `train` alone, and use
`handle_unknown='ignore'` so an unexpected value in val/test degrades gracefully (all-zero
row) instead of crashing — exactly how a production model must handle a category it never saw
during training. See Step 6 for the full "why train only" reasoning.


In [10]:
cat_cols = ['Animal_Type', 'Gender']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(splits['train'][cat_cols])
ohe_feature_names = ohe.get_feature_names_out(cat_cols)

for name in splits:
    part = splits[name]
    encoded = pd.DataFrame(ohe.transform(part[cat_cols]), columns=ohe_feature_names, index=part.index)
    splits[name] = pd.concat([part.drop(columns=cat_cols), encoded], axis=1)

print(f'One-hot expanded {cat_cols} into {len(ohe_feature_names)} columns')


One-hot expanded ['Animal_Type', 'Gender'] into 4 columns


## Step 6: Scale Numeric Features — and why everything above says "fit on train only"

**Why scale at all?** `Heart_Rate` ranges in the hundreds, `Age` ranges in the teens. Some ML
algorithms would otherwise treat a change of "10" in Heart Rate the same as a change of "10"
in Age — even though those mean very different things. `StandardScaler` rescales every numeric
column to a common range (average ≈ 0, spread ≈ 1) so no single feature dominates just
because of the units it happens to be measured in.

**Why "fit on train only" — the full explanation:** imagine studying for an exam. Train data
is your practice questions; test data is the real exam, which you're not supposed to see
beforehand. If your studying secretly involved peeking at the real exam questions, you'd score
great — but only because you cheated, not because you actually learned anything. That score
would lie to you about how you'd do on a genuinely new exam.

Some steps calculate a number *from* the data — e.g. "the average Heart Rate across these rows
is 118." If that average is calculated using all 2,200 rows (train+val+test mixed together),
then the val/test rows have quietly influenced a number that gets baked into training — that's
the cheating. This is called **data leakage**. The fix: calculate any such number using
**only** the train pile, then apply that same fixed number to val/test unchanged — val/test
never gets to influence the calculation, only receive the same treatment afterward.

(This is why the symptom vocabulary and the `"3 weeks" → 21` duration parsing in Steps 2-3
didn't need this treatment — those aren't estimates calculated *from* our specific sample of
rows, they're fixed facts true regardless of which rows we happened to have. The `Breed`
check in Step 4 is different again — that was run on the full dataset because it's an
exploratory question about the data itself, not a transform baked into the pipeline. Only
genuine statistical estimates that become part of the pipeline — averages, "which categories
exist" — need the train-only rule.)


In [11]:
numeric_cols = ['Age', 'Weight', 'Body_Temperature', 'Heart_Rate', 'Duration_Days', 'Symptom_Count']
scaler = StandardScaler()
scaler.fit(splits['train'][numeric_cols])

for name in splits:
    splits[name][numeric_cols] = scaler.transform(splits[name][numeric_cols])

print('Scaled numeric columns — train mean (~0) / std (~1):')
print(splits['train'][numeric_cols].agg(['mean', 'std']).round(3))


Scaled numeric columns — train mean (~0) / std (~1):
      Age  Weight  Body_Temperature  Heart_Rate  Duration_Days  Symptom_Count
mean  0.0    -0.0              -0.0        -0.0            0.0           -0.0
std   1.0     1.0               1.0         1.0            1.0            1.0


## Step 7: Label Encoding

The two answer columns (`Disease_Category`, `Urgency`) are still text. We add a numeric
version of each (`Disease_Category_Label`, `Urgency_Label`) because the training code and
MLflow logging need numbers — but we keep the original text columns too, since we'll want
readable labels when looking at results later.

**Not built here:** the "soft cascade" — Model B (Urgency) receiving Model A's (Condition)
*predicted probabilities* as input, per the architecture decision in `DECISIONS.md`. That
needs a trained Model A to exist first, so it belongs in the training step, not here.


In [12]:
cat_encoder = LabelEncoder().fit(splits['train']['Disease_Category'])
urg_encoder = LabelEncoder().fit(splits['train']['Urgency'])

for name in splits:
    part = splits[name]
    part['Disease_Category_Label'] = cat_encoder.transform(part['Disease_Category'])
    part['Urgency_Label'] = urg_encoder.transform(part['Urgency'])
    splits[name] = part

print('Disease_Category classes:', list(cat_encoder.classes_))
print('Urgency classes:', list(urg_encoder.classes_))


Disease_Category classes: ['Bacterial / Parasitic', 'Dental / Oral', 'Endocrine / Metabolic', 'Eye / Ear', 'Gastrointestinal', 'Musculoskeletal', 'Renal / Urinary', 'Respiratory', 'Skin / Fungal', 'Trauma / Poisoning', 'Viral Systemic']
Urgency classes: ['Emergency', 'Monitor', 'Okay']


## Save

In [13]:
for name in splits:
    out_path = f'../data/processed/{name}.csv'
    splits[name].to_csv(out_path, index=False)
    print(f'Saved {name}: {splits[name].shape} -> {out_path}')


Saved train: (1540, 45) -> ../data/processed/train.csv
Saved val: (330, 45) -> ../data/processed/val.csv
Saved test: (330, 45) -> ../data/processed/test.csv


---
## Summary

`data/processed/train.csv` (1,540 rows), `val.csv` (330), `test.csv` (330) — same 2,200 cases
as before, split into three stratified piles and reshaped into 45 number-only columns:
31 multi-hot symptom flags + `Symptom_Count`, one-hot `Animal_Type`/`Gender` (4 columns),
6 scaled numeric features, plus both label columns (text + numeric) for each target.
`Breed` was checked with a statistical association test and dropped — see Step 4 and
ADR-014.

Stratified splitting preserved Urgency proportions across train/val/test to within ~1%.

**Next:** `src/pettriage/data/preprocess.py` — port this exact recipe (split key, symptom
vocabulary, duration parser, fitted encoders/scaler) into reusable, importable code so
training and inference use identical transforms.
